# SCE-Net One-Class Compatibility (VAE) на позитивных парах

Этот ноутбук — практическая адаптация `sce_net_fashion_compatibility_pair_classification.ipynb` под **anomaly detection / one-class classification**.

Ключевая идея: обучаемся **только на хороших (y=1) парах** и считаем, что плохие пары — это отклонения от распределения хороших.


## Что переиспользуем из вашего baseline

Мы сохраняем максимально возможную логику из базового ноутбука:
- CLIP-эмбеддинги и загрузка модели через HuggingFace;
- SCE-Net блок: `condition_masks` + `weight_branch` для контекстного представления пары;
- конфиг-паттерн через `@dataclass`;
- RAM-cache загрузки изображений;
- пайплайн train/val/test и подбор порога на валидации.

Что меняем:
- вместо бинарного классификатора (`BCE`) добавляем **VAE-head** над парным представлением;
- train/val содержат только позитивные пары;
- score на инференсе = anomaly score (комбинация reconstruction + KL), затем порог.


In [ ]:
import os
import random
import math
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoModel, AutoProcessor
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score


## 1) Конфиг (оставляем стиль baseline) + фиксируем энкодер

По вашему требованию заменяем энкодер на:
`hf_model_name: str = "patrickjohncyh/fashion-clip"`.


In [ ]:
@dataclass
class CFG:
    seed: int = 42
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Data
    image_root: str = './images'
    train_pairs_path: str = './train_pairs.csv'   # только y=1
    val_pairs_path: str = './val_pairs.csv'       # только y=1
    test_pairs_path: str = './test_pairs.csv'     # y in {0,1} для подбора/оценки

    # Backbone
    hf_model_name: str = 'patrickjohncyh/fashion-clip'
    clip_emb_dim: int = 512
    freeze_clip: bool = True

    # SCE-Net
    condition_masks: int = 8
    hidden_dim: int = 512
    proj_dim: int = 256

    # VAE
    latent_dim: int = 64
    beta_kl: float = 0.2
    recon_loss: str = 'mse'

    # Training
    batch_size: int = 64
    num_workers: int = 4
    epochs: int = 10
    lr: float = 2e-4
    wd: float = 1e-4

    # Regularization from SCE-Net paper spirit
    l1_mask_lambda: float = 1e-5
    l2_embed_lambda: float = 1e-6

cfg = CFG()

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(cfg.seed)
print(cfg)


## 2) Подготовка данных: train/val только позитивы

Ожидается формат CSV: `sku1, sku2, y`.
- Для train/val оставляем только `y=1`.
- Для test сохраняем обе метки, чтобы валидировать порог.


In [ ]:
def load_pairs(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = {'sku1', 'sku2', 'y'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'Missing columns {missing} in {path}')
    return df

train_df = load_pairs(cfg.train_pairs_path)
val_df = load_pairs(cfg.val_pairs_path)
test_df = load_pairs(cfg.test_pairs_path)

train_pos = train_df[train_df.y == 1].reset_index(drop=True)
val_pos = val_df[val_df.y == 1].reset_index(drop=True)

print('train all:', len(train_df), 'train pos:', len(train_pos))
print('val all:', len(val_df), 'val pos:', len(val_pos))
print('test all:', len(test_df), 'test y distribution:', test_df.y.value_counts().to_dict())


## 3) RAM cache изображений (переиспользование baseline)


In [ ]:
class ImageRAMCache:
    def __init__(self, image_root: str):
        self.image_root = image_root
        self.cache: Dict[str, Image.Image] = {}

    def _resolve_path(self, sku: str) -> str:
        return os.path.join(self.image_root, f'{sku}.jpg')

    def load_one(self, sku: str) -> Image.Image:
        if sku in self.cache:
            return self.cache[sku]
        path = self._resolve_path(sku)
        img = Image.open(path).convert('RGB')
        self.cache[sku] = img
        return img

    def preload(self, skus: List[str], max_workers: int = 16):
        uniq = list(dict.fromkeys(skus))
        def _load(s):
            try:
                self.load_one(s)
            except Exception:
                pass
        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            list(ex.map(_load, uniq))

cache = ImageRAMCache(cfg.image_root)
all_skus = pd.concat([
    train_pos[['sku1', 'sku2']], val_pos[['sku1', 'sku2']], test_df[['sku1', 'sku2']]
], axis=0).stack().astype(str).tolist()
cache.preload(all_skus)
print('Cached images:', len(cache.cache))


## 4) Dataset/Collate для пар


In [ ]:
processor = AutoProcessor.from_pretrained(cfg.hf_model_name)

class PairDataset(Dataset):
    def __init__(self, df: pd.DataFrame, cache: ImageRAMCache):
        self.df = df.reset_index(drop=True)
        self.cache = cache

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sku1, sku2, y = str(row.sku1), str(row.sku2), int(row.y)
        img1 = self.cache.load_one(sku1)
        img2 = self.cache.load_one(sku2)
        return img1, img2, y, sku1, sku2

def collate_pairs(batch):
    imgs1, imgs2, ys, sku1s, sku2s = zip(*batch)
    p1 = processor(images=list(imgs1), return_tensors='pt')
    p2 = processor(images=list(imgs2), return_tensors='pt')
    return p1, p2, torch.tensor(ys, dtype=torch.long), list(sku1s), list(sku2s)


## 5) SCE-Net encoder + VAE head

Как ложится VAE на задачу:
1. Строим контекстные эмбеддинги пары через SCE-Net (маски + weight branch).
2. Из финального pair representation получаем `mu, logvar`.
3. Сэмплируем `z = mu + eps * sigma` (reparameterization).
4. Декодируем `z` обратно в пространство pair representation.
5. Учим на позитивных парах: минимизируем reconstruction + beta * KL.

На инференсе аномалия = высокий reconstruction/ELBO score.


## 5.1) Интуиция VAE максимально подробно

### Что такое `mu` и `logvar`
Для каждой пары товаров сеть предсказывает параметры гауссовского распределения в латентном пространстве:
- `mu` — центр (среднее) латентного распределения для текущей пары;
- `logvar` — логарифм дисперсии (`log(σ²)`), а не сама дисперсия.

Почему именно `logvar`:
- численно стабильнее обучать (дисперсия всегда положительна);
- легко получить стандартное отклонение: `std = exp(0.5 * logvar)`.

Идея: вместо «жесткой точки» в латентном пространстве мы учим **распределение** для хорошей пары.
Это позволяет модели описывать естественную вариативность совместимых образов.

### Reparameterization trick (пере-параметризация)
Если просто сэмплировать `z ~ N(mu, sigma^2)`, градиент через случайную выборку не пройдет.
Поэтому используем разложение:
`z = mu + eps * std`, где `eps ~ N(0, I)`.

Что это дает:
- случайность уходит в `eps` (внешний шум);
- `mu` и `std` становятся дифференцируемыми параметрами;
- модель можно обучать обычным backprop.

### Что делает decoder
Decoder получает `z` и восстанавливает исходное компактное представление пары (`pair_repr`) из SCE-Net.

- Если пара «похожа» на train-позитивы, decoder обычно восстанавливает ее хорошо (малый reconstruction error).
- Если пара out-of-distribution (плохая сочетаемость), reconstruction error растет.

То есть decoder — это механизм проверки «насколько пара лежит на manifold хороших сочетаний».

### Почему в loss есть KL-терм
`KL(q(z|x) || p(z))` (обычно к `N(0, I)`) не дает энкодеру выучить произвольные разрозненные острова.
Он «собирает» латентное пространство в более гладкое, обобщающее распределение.

Итоговый компромисс:
- reconstruction учит точности восстановления;
- KL учит регулярности и обобщению.

В one-class сценарии это критично: нам нужно не запомнить только train-пары, а адекватно распознавать отклонения.



In [ ]:
class SCENetVAE(nn.Module):
    def __init__(self, cfg: CFG):
        super().__init__()
        self.cfg = cfg
        self.clip = AutoModel.from_pretrained(cfg.hf_model_name)
        if cfg.freeze_clip:
            for p in self.clip.parameters():
                p.requires_grad = False

        d = cfg.clip_emb_dim
        m = cfg.condition_masks

        self.condition_masks = nn.Parameter(torch.randn(m, d) * 0.02)
        self.weight_branch = nn.Sequential(
            nn.Linear(2 * d, cfg.hidden_dim), nn.ReLU(),
            nn.Linear(cfg.hidden_dim, cfg.hidden_dim), nn.ReLU(),
            nn.Linear(cfg.hidden_dim, m)
        )

        self.pair_proj = nn.Sequential(
            nn.Linear(2 * d, cfg.hidden_dim), nn.ReLU(),
            nn.Linear(cfg.hidden_dim, cfg.proj_dim)
        )

        self.mu_head = nn.Linear(cfg.proj_dim, cfg.latent_dim)
        self.logvar_head = nn.Linear(cfg.proj_dim, cfg.latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(cfg.latent_dim, cfg.hidden_dim), nn.ReLU(),
            nn.Linear(cfg.hidden_dim, cfg.proj_dim)
        )

    def _clip_embed(self, pixel_values):
        out = self.clip.get_image_features(pixel_values=pixel_values)
        return F.normalize(out, dim=-1)

    def _sce_contextualize(self, v1, v2):
        w_logits = self.weight_branch(torch.cat([v1, v2], dim=-1))
        w = torch.softmax(w_logits, dim=-1)

        e1_all = v1.unsqueeze(1) * self.condition_masks.unsqueeze(0)
        e2_all = v2.unsqueeze(1) * self.condition_masks.unsqueeze(0)

        e1 = torch.einsum('bm,bmd->bd', w, e1_all)
        e2 = torch.einsum('bm,bmd->bd', w, e2_all)

        pair_repr = self.pair_proj(torch.cat([e1, e2], dim=-1))
        return pair_repr, e1, e2, w

    def forward(self, p1, p2):
        v1 = self._clip_embed(p1)
        v2 = self._clip_embed(p2)
        pair_repr, e1, e2, w = self._sce_contextualize(v1, v2)

        # mu: центр латентного распределения q(z|x) для каждой пары
        mu = self.mu_head(pair_repr)
        # logvar: log(σ²), используем лог-дисперсию для численной стабильности
        logvar = self.logvar_head(pair_repr)
        # std = σ = exp(0.5 * logvar)
        std = torch.exp(0.5 * logvar)
        # eps ~ N(0, I), независимый шум
        eps = torch.randn_like(std)
        # Reparameterization trick: z = mu + eps * std (градиент проходит в mu/logvar)
        z = mu + eps * std
        # decoder восстанавливает pair_repr из латентного z
        recon = self.decoder(z)

        return {
            'pair_repr': pair_repr,
            'recon': recon,
            'mu': mu,
            'logvar': logvar,
            'e1': e1,
            'e2': e2,
            'w': w,
        }


## 6) Loss для one-class VAE


In [ ]:
def vae_loss_fn(out, cfg: CFG):
    x = out['pair_repr']
    recon = out['recon']
    mu, logvar = out['mu'], out['logvar']

    if cfg.recon_loss == 'mse':
        recon_loss = F.mse_loss(recon, x, reduction='mean')
    else:
        recon_loss = F.smooth_l1_loss(recon, x, reduction='mean')

    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    l1_masks = out['w'].new_tensor(0.0)
    l2_embed = out['w'].new_tensor(0.0)

    total = recon_loss + cfg.beta_kl * kl + cfg.l1_mask_lambda * l1_masks + cfg.l2_embed_lambda * l2_embed
    return total, {'recon': recon_loss.item(), 'kl': kl.item()}

def anomaly_score(out):
    rec = ((out['recon'] - out['pair_repr']) ** 2).mean(dim=-1)
    kl = -0.5 * torch.mean(1 + out['logvar'] - out['mu'].pow(2) - out['logvar'].exp(), dim=-1)
    return rec + 0.2 * kl


## 7) Обучение и подбор порога


In [ ]:
train_loader = DataLoader(PairDataset(train_pos, cache), batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, collate_fn=collate_pairs)
val_loader = DataLoader(PairDataset(val_pos, cache), batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, collate_fn=collate_pairs)
test_loader = DataLoader(PairDataset(test_df, cache), batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, collate_fn=collate_pairs)

model = SCENetVAE(cfg).to(cfg.device)
opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)

best_val = float('inf')
best_state = None

for epoch in range(cfg.epochs):
    model.train()
    train_losses = []
    for p1, p2, y, *_ in train_loader:
        p1 = {k: v.to(cfg.device) for k, v in p1.items()}
        p2 = {k: v.to(cfg.device) for k, v in p2.items()}
        out = model(p1['pixel_values'], p2['pixel_values'])
        loss, logs = vae_loss_fn(out, cfg)
        opt.zero_grad()
        loss.backward()
        opt.step()
        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    with torch.no_grad():
        for p1, p2, y, *_ in val_loader:
            p1 = {k: v.to(cfg.device) for k, v in p1.items()}
            p2 = {k: v.to(cfg.device) for k, v in p2.items()}
            out = model(p1['pixel_values'], p2['pixel_values'])
            loss, _ = vae_loss_fn(out, cfg)
            val_losses.append(loss.item())

    tr, va = float(np.mean(train_losses)), float(np.mean(val_losses))
    print(f'Epoch {epoch+1}/{cfg.epochs} | train={tr:.5f} | val={va:.5f}')
    if va < best_val:
        best_val = va
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}

torch.save(best_state, 'sce_net_oneclass_vae_best.pt')
print('Best val:', best_val)


## 8) Оценка на test (поиск порога и метрики)


In [ ]:
model.load_state_dict(torch.load('sce_net_oneclass_vae_best.pt', map_location='cpu'))
model.to(cfg.device).eval()

scores, ys = [], []
with torch.no_grad():
    for p1, p2, y, *_ in test_loader:
        p1 = {k: v.to(cfg.device) for k, v in p1.items()}
        p2 = {k: v.to(cfg.device) for k, v in p2.items()}
        out = model(p1['pixel_values'], p2['pixel_values'])
        s = anomaly_score(out).detach().cpu().numpy()
        scores.extend(s.tolist())
        ys.extend(y.numpy().tolist())

scores = np.array(scores)
ys = np.array(ys)

# y=1 good, y=0 bad; для anomaly bad должен иметь высокий score
bad_target = (ys == 0).astype(int)
auc = roc_auc_score(bad_target, scores)
ap = average_precision_score(bad_target, scores)

thr_grid = np.quantile(scores, np.linspace(0.05, 0.95, 200))
best = None
for t in thr_grid:
    pred_bad = (scores >= t).astype(int)
    f1 = f1_score(bad_target, pred_bad)
    if best is None or f1 > best[0]:
        best = (f1, t, precision_score(bad_target, pred_bad), recall_score(bad_target, pred_bad))

print({'AUC_bad': auc, 'AP_bad': ap, 'best_f1_bad': best[0], 'thr': best[1], 'precision': best[2], 'recall': best[3]})


## 9) Подробная методологическая часть: как VAE из статьи использовать в вашей постановке

1. **Почему one-class здесь уместен**: если негативы зашумлены/неестественны, дискриминативная граница учится на артефактах генерации негативов.
2. **Что моделирует VAE**: плотность/многообразие *хороших* сочетаний в пространстве SCE-Net pair representation.
3. **Как интерпретировать score**:
   - низкий score: пара похожа на распределение эталонных образов;
   - высокий score: OOD/аномалия относительно позитивного manifold.
4. **Практический рецепт**:
   - train/val: только y=1;
   - threshold tuning: на отдельном holdout/test с метками стилиста;
   - мониторить `FPR@TPR`, PR-AUC по bad-классу.
5. **Что подкрутить в первую очередь**:
   - `condition_masks` (6/8/12/16), `latent_dim` (32/64/128), `beta_kl` (0.05..1.0);
   - тип recon loss (`mse` vs `smooth_l1`);
   - partial unfreeze последних слоёв CLIP, если домен сильно отличается.
6. **Важный момент про порог**:
   - использовать бизнес-метрику: например, лучше выше precision по bad, чтобы не отбрасывать хорошие базовые пары типа white tee + jeans.
7. **Гибридная стратегия (рекомендуется)**:
   - stage-1 one-class VAE как фильтр аномалий;
   - stage-2 лёгкий calibrator/ранжирование на небольшом ручном наборе hard-negatives.

8. **Практическая интерпретация `mu/logvar` для продакшна**:
   - `mu` можно считать «координатой стиля» пары;
   - высокая `logvar` часто сигнализирует, что модель не уверена в том, где расположить пару в manifold;
   - пары с одновременно высоким reconstruction error и высоким KL — самые вероятные аномалии.
9. **Почему это помогает с кейсом "white tee + jeans"**:
   - если такие пары часто есть среди позитивных образов, VAE учит их как нормальную область плотности;
   - без искусственных hard-negative эвристик вероятность ложного «bad» снижается.
